In [37]:
import pandas as pd

df = pd.read_csv("../data/raw/reexposition_finess/finess_etablissements.txt", sep=";", encoding="utf-8")
print(df.head())

/tmp/ipykernel_6826/2209818447.py:3: DtypeWarning: Columns (0,1,12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/reexposition_finess/finess_etablissements.txt", sep=";", encoding="utf-8")


  nofinesset nofinessej                           rs  \
0   10000024   10780054               CH DE FLEYRIAT   
1   10000032   10780062                 CH BUGEY SUD   
2   10000065   10780096  CH DE TREVOUX - MONTPENSIER   
3   10000081   10780112            CH DU PAYS DE GEX   
4   10000099   10780120              CH DE MEXIMIEUX   

                                         rslongue complrs compldistrib  \
0  CENTRE HOSPITALIER DE BOURG-EN-BRESSE FLEYRIAT     NaN          NaN   
1                    CENTRE HOSPITALIER BUGEY SUD     NaN          NaN   
2     CENTRE HOSPITALIER DE TREVOUX - MONTPENSIER     NaN          NaN   
3               CENTRE HOSPITALIER DU PAYS DE GEX     NaN          NaN   
4                 CENTRE HOSPITALIER DE MEXIMIEUX     NaN          NaN   

   numvoie typvoie              voie compvoie  ... codesph  \
0    900.0     RTE          DE PARIS      NaN  ...     1.0   
1    700.0      AV         DE NARVIK      NaN  ...     1.0   
2     14.0       R      DE L'HOP

# Nettoyage de la table etablissements en SQL sur Big Query

CREATE OR REPLACE TABLE `sante-et-territoires.finess.finess_etablissements_clean` AS
SELECT
  SAFE_CAST(nofinesset AS STRING) AS nofinesset,
  SAFE_CAST(nofinessej AS STRING) AS nofinessej,
  TRIM(rs) AS rs,
  TRIM(rslongue) AS rslongue,
  NULLIF(TRIM(complrs), '') AS complrs,
  numvoie,
  NULLIF(TRIM(typvoie), '') AS typvoie,
  NULLIF(TRIM(voie), '') AS voie,
  NULLIF(TRIM(compvoie), '') AS compvoie,
  NULLIF(TRIM(compldistrib), '') AS compldistrib,
  NULLIF(TRIM(lieuditbp), '') AS lieuditbp,

  -- commune est déjà un entier → on le laisse tel quel
  commune,

  TRIM(ligneacheminement) AS ligneacheminement,
  departement,
  libdepartement,
  categetab,
  libcategetab,
  siret,
  codeape,
  coordxet,
  coordyet,
  SAFE_CAST(dateouv AS DATE) AS dateouv,
  SAFE_CAST(dateautor AS DATE) AS dateautor

FROM `sante-et-territoires.finess.etablissements`

-- Ici on déduplique sur nofinesset (clé établissement géographique)
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY nofinesset
    ORDER BY dateouv DESC
) = 1;

# Nettoyage de la table activites en SQL sur Big Query
CREATE OR REPLACE TABLE `sante-et-territoires.finess.finess_activites_soin_clean` AS
SELECT
  SAFE_CAST(nofinessej AS STRING) AS nofinessej,
  nofinesset,
  TRIM(rsej) AS rsej,
  activite,
  TRIM(libactivite) AS libactivite,
  forme,
  TRIM(libforme) AS libforme,
  modalite,
  TRIM(libmodalite) AS libmodalite,

  SAFE_CAST(datemeo AS DATE) AS datemeo,
  SAFE_CAST(datefin AS DATE) AS datefin

FROM `sante-et-territoires.finess.activites_soin`

# nettoyage table etablissements sociaux 
CREATE OR REPLACE TABLE `sante-et-territoires.finess.equipements_sociaux_clean` AS
SELECT
  nofinessej,
  nofinesset,
  de AS de_equipement,
  libde AS libde_equipement,
  libta AS libta_equipement

FROM `sante-et-territoires.finess.equipements_sociaux`

# jointure etbalisements et activites soins
CREATE OR REPLACE TABLE `sante-et-territoires.finess.table_finale` AS
SELECT e.nofinesset AS nofinesset,
a.rsej AS rsej,
a.activite AS activite,
a.libactivite AS libactivite,
a.forme AS forme,
a.libforme AS libforme,
e.nofinessej AS nofinessej,
e.rs AS rs,
e.rslongue AS rslongue,
e.commune AS commune,
e.departement AS departement,
e.libdepartement AS libdepartement,
e.categetab AS categetab,
e.libcategetab AS libcategetab,
e.coordyet AS coodyet,
e.coordxet AS coodxet
FROM `sante-et-territoires.finess.finess_activites_soin_clean` AS a
FULL OUTER JOIN `sante-et-territoires.finess.finess_etablissements_clean` AS e
ON e.nofinesset = a.nofinesset

# jointure equipements sociaux et table finale
SELECT *
FROM `sante-et-territoires.finess.equipements_sociaux_clean` AS s
FULL OUTER JOIN `sante-et-territoires.finess.table_finale` AS t
ON t.nofinesset = s.nofinesset

# Importation des donneés dans Python 

In [38]:
from google.cloud import bigquery
import os


# Option A : Via variable d'environnement (Recommandé) 
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../data/auth/sante-et-territoires-216daad01024.json" 
client = bigquery.Client() 
# Option B : Directement dans le code# 
#client = bigquery.Client.from_service_account_json("C:\\Users\stefl\Documents\Projet\sante-et-territoires-216daad01024.json")
#export GOOGLE_APPLICATION_CREDENTIALS="C:\Users\stefl\Documents\Projet\sante-et-territoires-216daad01024.json"

#client = bigquery.Client()

query = """
SELECT *
FROM `sante-et-territoires.finess.table_finale_3`
"""

df = client.query(query).to_dataframe()
print(df)

       nofinessej nofinesset de_equipement  \
0            None       None          None   
1       010783009  010012094           924   
2       010790368  010007078           657   
3       010003929  010003978           963   
4       010003929  010003978           657   
...           ...        ...           ...   
168375       None       None          None   
168376       None       None          None   
168377  780026118  780026126           469   
168378  780026118  780026126           469   
168379       None       None          None   

                                         libde_equipement  \
0                                                    None   
1                            Accueil pour Personnes Âgées   
2                 Accueil temporaire pour Personnes Âgées   
3       Plateforme d'accompagnement et de répit des ai...   
4                 Accueil temporaire pour Personnes Âgées   
...                                                   ...   
168375              

In [39]:
# Regroupement des catégories de types d'établissements
def regrouper_categorie(cat):
    cat = cat.lower()

    if any(x in cat for x in ["hospital", "clinique", "soins", "dialyse", "ssr", "had", "cancer"]):
        return "Hopitaux cliniques"

    if any(x in cat for x in ["handicap", "ime", "itep", "mas", "fam", "esat", "sensoriel", "e.s.a.t.", "i.m.e.", "i.t.e.p.", "m.a.s.", "s.a.v.s.", "foyer de vie", "entreprise adaptée", "esat", "institut d'éducation motrice", "service mandataire judiciaire à la protection des majeurs", "c.a.m.s.p."]):
        return "Médico-social handicap"

    if any(x in cat for x in ["ehpad", "ehpa", "autonomie", "personnes âgées", "longue durée", "personnes agées"]):
        return "Personnes âgées"

    if any(x in cat for x in ["chrs", "casa", "c.a.d.a", "hébergement", "foyer", "maison relais"]):
        return "Social / Hébergement"

    if any(x in cat for x in ["pharmacie", "centre de santé", "maison de santé", "laboratoire", "c.m.p.", "cabinet", "infirmier", "kinésithérapeute", "dentiste", "opticien", "structure dispensatrice à domicile d'oxygène à usage médical", "maison médicale de garde (MMG)"]):
        return "Soins de ville"

    if any(x in cat for x in ['prévention', "vaccination", "dépistage", "csapa", "caarud" ]):
        return "Prévention / Santé publique"

    if any(x in cat for x in ["enfance", "camsp", "aemo", "aed", "pouponnière", "maison d'enfants", "educative", "protection maternelle et infantile", "protection infantile", "pmi", "p.m.i."]):
        return "Enfance / Protection"

    if any(x in cat for x in ["cpts", "mdph", "coordination", "groupement"]):
        return "Coordination / Administration"
    if any(x in cat for x in ["centre d'accueil", "accueil"]):
            return "Centres d'accueil"
    if any(x in cat for x in ["ecoles"]):
        return "Ecoles"
    return "Autres"

# Application
df["groupe"] = df["libcategetab"].apply(regrouper_categorie)

In [40]:
# Filtrer les établissements situés en France métropolitaine
df = df[df["latitude"].between(35, 54)]
df = df[df["longitude"].between(-10, 10)]

In [41]:
df[df.duplicated()]

,nofinessej,nofinesset,de_equipement,libde_equipement,libta_equipement,nofinesset_1,rsej,activite,libactivite,forme,...,commune,departement,libdepartement,categetab,libcategetab,coodyet,coodxet,latitude,longitude,groupe
20,010013704,010013712,959,"Hébergement d'Urgence Adultes, Familles Diffic...",Hébergement de Nuit Eclaté,010013712,None,None,None,None,...,053,01,AIN,219,Autre Centre d'Accueil,6569378.6,871811.5,46.202431,5.228481,Centres d'accueil
24,010789600,010013720,959,"Hébergement d'Urgence Adultes, Familles Diffic...",Hébergement de Nuit Eclaté,010013720,None,None,None,None,...,053,01,AIN,219,Autre Centre d'Accueil,6568244.7,871218.8,46.192375,5.220384,Centres d'accueil
49,940031339,010005619,900,Action Médico-Sociale Précoce,Accueil de jour et accompagnement en milieu or...,010005619,None,None,None,None,...,004,01,AIN,190,Centre Action Médico-Sociale Précoce (C.A.M.S.P.),6542212.5,880861.6,45.955551,5.335380,Médico-social handicap
102,None,None,None,None,None,010000024,CH FLEYRIAT,03,"Gynécologie, obstétrique, néonatologie, réanim...",01,...,451,01,AIN,355,Centre Hospitalier (C.H.),6571540.8,870262.2,46.222286,5.209181,Hopitaux cliniques
104,None,None,None,None,None,010000024,CH FLEYRIAT,03,"Gynécologie, obstétrique, néonatologie, réanim...",01,...,451,01,AIN,355,Centre Hospitalier (C.H.),6571540.8,870262.2,46.222286,5.209181,Hopitaux cliniques
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168332,780803821,780804050,358,Soins infirmiers à Domicile,Prestation en milieu ordinaire,780804050,None,None,None,None,...,440,78,YVELINES,354,Service de Soins Infirmiers A Domicile (S.S.I....,6877586.0,620221.3,48.993243,1.909814,Hopitaux cliniques
168336,780016820,780804100,358,Soins infirmiers à Domicile,Prestation en milieu ordinaire,780804100,None,None,None,None,...,650,78,YVELINES,354,Service de Soins Infirmiers A Domicile (S.S.I....,6865003.2,636165.9,48.881880,2.129574,Hopitaux cliniques
168341,780008868,780008918,358,Soins infirmiers à Domicile,Prestation en milieu ordinaire,780008918,None,None,None,None,...,640,78,YVELINES,354,Service de Soins Infirmiers A Domicile (S.S.I....,6853931.0,639096.8,48.782593,2.171121,Hopitaux cliniques
168364,None,None,None,None,None,780001558,ASS DIALYSE A DOMICILE (ADDY),16,Traitement de l'insuffisance rénale chronique ...,14,...,686,78,YVELINES,146,Structure d'Alternative à la dialyse en centre,6856321.8,639418.8,48.804124,2.175163,Hopitaux cliniques


In [42]:
#je remplace les valeurs manquantes de nofinesset par les valeurs de nofinesset_1
df["nofinesset"] = df["nofinesset_1"].fillna(df["nofinesset"])
df.head()

,nofinessej,nofinesset,de_equipement,libde_equipement,libta_equipement,nofinesset_1,rsej,activite,libactivite,forme,...,commune,departement,libdepartement,categetab,libcategetab,coodyet,coodxet,latitude,longitude,groupe
0,None,010786762,None,None,None,010786762,None,None,None,None,...,053,01,AIN,330,Ecoles Formant aux Professions Sociales,6569218.9,870665.3,46.201284,5.213564,Ecoles
1,010783009,010012094,924,Accueil pour Personnes Âgées,Accueil de Jour,010012094,None,None,None,None,...,281,01,AIN,502,EHPA ne percevant pas des crédits d'assurance ...,6578454.4,939656.3,46.263454,6.112319,Personnes âgées
2,010790368,010007078,657,Accueil temporaire pour Personnes Âgées,Accueil de Jour,010007078,None,None,None,None,...,320,01,AIN,207,Centre de Jour pour Personnes Agées,6580892.8,844320.2,46.312512,4.875599,Personnes âgées
3,010003929,010003978,963,Plateforme d'accompagnement et de répit des ai...,Accueil de Jour,010003978,None,None,None,None,...,322,01,AIN,207,Centre de Jour pour Personnes Agées,6538381.2,840594.6,45.930589,4.814504,Personnes âgées
4,010003929,010003978,657,Accueil temporaire pour Personnes Âgées,Accueil de Jour,010003978,None,None,None,None,...,322,01,AIN,207,Centre de Jour pour Personnes Agées,6538381.2,840594.6,45.930589,4.814504,Personnes âgées


In [43]:
#nettoyage des doublons
df_clean = df.drop_duplicates()

In [44]:
#je crée un colonne code_insee à partir de la colonne code commune et de la colonne departement
df_clean["code_insee"] = df_clean["departement"].astype(str) + df_clean["commune"].astype(str)
df_clean.head()

/tmp/ipykernel_6826/597779604.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean["code_insee"] = df_clean["departement"].astype(str) + df_clean["commune"].astype(str)


,nofinessej,nofinesset,de_equipement,libde_equipement,libta_equipement,nofinesset_1,rsej,activite,libactivite,forme,...,departement,libdepartement,categetab,libcategetab,coodyet,coodxet,latitude,longitude,groupe,code_insee
0,None,010786762,None,None,None,010786762,None,None,None,None,...,01,AIN,330,Ecoles Formant aux Professions Sociales,6569218.9,870665.3,46.201284,5.213564,Ecoles,01053
1,010783009,010012094,924,Accueil pour Personnes Âgées,Accueil de Jour,010012094,None,None,None,None,...,01,AIN,502,EHPA ne percevant pas des crédits d'assurance ...,6578454.4,939656.3,46.263454,6.112319,Personnes âgées,01281
2,010790368,010007078,657,Accueil temporaire pour Personnes Âgées,Accueil de Jour,010007078,None,None,None,None,...,01,AIN,207,Centre de Jour pour Personnes Agées,6580892.8,844320.2,46.312512,4.875599,Personnes âgées,01320
3,010003929,010003978,963,Plateforme d'accompagnement et de répit des ai...,Accueil de Jour,010003978,None,None,None,None,...,01,AIN,207,Centre de Jour pour Personnes Agées,6538381.2,840594.6,45.930589,4.814504,Personnes âgées,01322
4,010003929,010003978,657,Accueil temporaire pour Personnes Âgées,Accueil de Jour,010003978,None,None,None,None,...,01,AIN,207,Centre de Jour pour Personnes Agées,6538381.2,840594.6,45.930589,4.814504,Personnes âgées,01322


#Renomage des colonnes 

In [45]:
df_clean = df_clean.rename(columns={
    "rs": "raison_sociale",
    "libcategetab": "categorie",
    "commune": "code commune",
    "nofinessej": "numero finess etablissement juridique",
      'nofinesset': "numero finess etablissement", 
      'de_equipement': "description equipement sociaux", 
      'libde_equipement': "libelle description equipement sociaux",
        'libactivite': "libelle activite",
        'libforme': "libelle forme",
        'libdepartement': "libelle departement",
         'groupe' : "type d etablissements",
})

In [46]:
#suppression des colonnes nofinesset_1, description equipement sociaux, coodyet, coodxet
df_clean = df_clean.drop(columns=["nofinesset_1", "description equipement sociaux",  "coodyet","coodxet"])

In [47]:
# j'écris le fichier en dur
df_clean.to_csv("../data/processed/finess_totale_clean_2.csv", index=False)

In [48]:
#je filtre les données sur les départements de la région Occitanie
departements_occitanie = ["09","11", "12", "30","31", "32", "34", "46", "48","65", "66", "81", "82",  ]
df_occitanie = df_clean[df_clean["departement"].isin(departements_occitanie)]
df_occitanie = df_occitanie.copy()
df_occitanie.to_csv("../data/processed/finess_occitanie2.csv", index=False)

In [49]:
# je crée une table jointe des données des communes et des données des établissements en occitanie
dtype_cols = {
    "code_insee": "string",
    "code_postal": "string",
    "dep_code": "string",
    "canton_code": "string",
    "epci_code": "string", 
    "code_insee_centre_zone_emploi": "string"
}

df_communes = pd.read_csv("../data/geo/communes-france-2025.csv", dtype=dtype_cols
)
df_join = pd.merge(df_occitanie, df_communes, left_on="code_insee", right_on="code_insee", how="left")
df_join.to_csv("../data/processed/finess_occitanie_join.csv", index=False)

In [50]:
df_join.head()

,numero finess etablissement juridique,numero finess etablissement,libelle description equipement sociaux,libta_equipement,rsej,activite,libelle activite,forme,libelle forme,nofinessej_1,...,code_unite_urbaine,taille_unite_urbaine,population,superficie_hectare,superficie_km2,densite,latitude_centre,longitude_centre,grille_densite,grille_densite_texte
0,None,090002585,None,None,None,None,None,None,None,090002577,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,090784125,None,None,AAIR MIDI PYRENEES,16,Traitement de l'insuffisance rénale chronique ...,14,Non saisonnier,310000633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,090002833,None,None,AAIR MIDI PYRENEES,16,Traitement de l'insuffisance rénale chronique ...,00,Pas de forme,310000633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,090002833,None,None,AAIR MIDI PYRENEES,16,Traitement de l'insuffisance rénale chronique ...,14,Non saisonnier,310000633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,090784679,None,None,AAIR MIDI PYRENEES,16,Traitement de l'insuffisance rénale chronique ...,00,Pas de forme,310000633,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
